# 3. Recurrent Neural Networks (RNN): designed for sequential data (uses loops to maintain memory)

# RNN THEORY:
RNNs are designed to handle sequential data where the order matters (time series, text, speech)
Key concept: RNNs have a "memory" - they remember information from previous time steps
At each time step t, RNN takes current input x_t and previous hidden state h_{t-1}
Formula: h_t = tanh(W_hh * h_{t-1} + W_xh * x_t + b_h)
         y_t = W_hy * h_t + b_y
Where:
- h_t: hidden state at time t (the "memory")
- W_hh: hidden-to-hidden weight matrix
- W_xh: input-to-hidden weight matrix  
- W_hy: hidden-to-output weight matrix
- b_h, b_y: bias terms

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

class ProductionRNN(nn.Module):
    """
    Production-ready RNN model with proper initialization, dropout, and layer normalization
    """
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, dropout=0.2):
        super(ProductionRNN, self).__init__()
        
        # Store hyperparameters for model reconstruction
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        
        # Multi-layer RNN with dropout for regularization
        # LSTM is used instead of vanilla RNN to solve vanishing gradient problem
        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,  # Dropout only between layers
            batch_first=True,  # Input shape: (batch, seq_len, features)
            bidirectional=False  # Can be set to True for bidirectional processing
        )
        
        # Layer normalization for stable training
        self.layer_norm = nn.LayerNorm(hidden_size)
        
        # Dropout for output regularization
        self.dropout = nn.Dropout(dropout)
        
        # Final linear layer to map hidden state to output
        self.fc = nn.Linear(hidden_size, output_size)
        
        # Initialize weights using Xavier initialization for better convergence
        self._init_weights()
    
    def _init_weights(self):
        """
        Initialize weights using Xavier/Glorot initialization
        This helps with gradient flow and faster convergence
        """
        for name, param in self.rnn.named_parameters():
            if 'weight_ih' in name:  # Input-to-hidden weights
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:  # Hidden-to-hidden weights
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:  # Bias terms
                param.data.fill_(0)
                # Set forget gate bias to 1 for LSTM (helps with long sequences)
                n = param.size(0)
                param.data[n//4:n//2].fill_(1)
        
        # Initialize final linear layer
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)
    
    def forward(self, x, hidden=None):
        """
        Forward pass through the RNN
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, input_size)
            hidden: Initial hidden state (optional)
        
        Returns:
            output: Final output tensor
            hidden: Final hidden state
        """
        batch_size = x.size(0)
        
        # Initialize hidden state if not provided
        if hidden is None:
            hidden = self._init_hidden(batch_size, x.device)
        
        # Pass through RNN layers
        # rnn_out shape: (batch_size, seq_len, hidden_size)
        # hidden contains (h_n, c_n) for LSTM
        rnn_out, hidden = self.rnn(x, hidden)
        
        # Apply layer normalization to stabilize training
        rnn_out = self.layer_norm(rnn_out)
        
        # Take the last time step output for sequence classification
        # For sequence-to-sequence tasks, you'd use all time steps
        last_output = rnn_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout for regularization
        last_output = self.dropout(last_output)
        
        # Map to output size through linear layer
        output = self.fc(last_output)  # Shape: (batch_size, output_size)
        
        return output, hidden
    
    def _init_hidden(self, batch_size, device):
        """
        Initialize hidden state with zeros
        For LSTM, we need both hidden state (h) and cell state (c)
        """
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)

class RNNTrainer:
    """
    Production-ready trainer class with proper logging, checkpointing, and validation
    """
    def __init__(self, model, criterion, optimizer, device):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.train_losses = []
        self.val_losses = []
    
    def train_epoch(self, train_loader):
        """
        Train for one epoch
        """
        self.model.train()  # Set model to training mode
        total_loss = 0
        num_batches = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            # Move data to device (GPU/CPU)
            data, target = data.to(self.device), target.to(self.device)
            
            # Zero gradients from previous iteration
            self.optimizer.zero_grad()
            
            # Forward pass: compute predictions
            output, _ = self.model(data)
            
            # Compute loss between predictions and targets
            loss = self.criterion(output, target)
            
            # Backward pass: compute gradients
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # Update weights using computed gradients
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches
        self.train_losses.append(avg_loss)
        return avg_loss
    
    def validate(self, val_loader):
        """
        Validate model performance
        """
        self.model.eval()  # Set model to evaluation mode
        total_loss = 0
        num_batches = 0
        
        with torch.no_grad():  # Disable gradient computation for efficiency
            for data, target in val_loader:
                data, target = data.to(self.device), target.to(self.device)
                
                # Forward pass only
                output, _ = self.model(data)
                loss = self.criterion(output, target)
                
                total_loss += loss.item()
                num_batches += 1
        
        avg_loss = total_loss / num_batches
        self.val_losses.append(avg_loss)
        return avg_loss
    
    def train(self, train_loader, val_loader, epochs, save_path=None):
        """
        Full training loop with validation and checkpointing
        """
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            # Train for one epoch
            train_loss = self.train_epoch(train_loader)
            
            # Validate
            val_loss = self.validate(val_loader)
            
            print(f'Epoch {epoch+1}/{epochs}:')
            print(f'  Train Loss: {train_loss:.4f}')
            print(f'  Val Loss: {val_loss:.4f}')
            
            # Save best model checkpoint
            if save_path and val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                }, save_path)
                print(f'  New best model saved!')

# Example usage for time series prediction
def create_sample_data(seq_len=50, num_samples=1000, input_size=1):
    """
    Create sample sequential data for demonstration
    This generates a simple sine wave with noise
    """
    # Generate time steps
    t = np.linspace(0, 4*np.pi, seq_len)
    
    # Create sequences
    X = []
    y = []
    
    for i in range(num_samples):
        # Generate sine wave with random phase and frequency
        phase = np.random.uniform(0, 2*np.pi)
        freq = np.random.uniform(0.5, 2.0)
        noise = np.random.normal(0, 0.1, seq_len)
        
        # Create sequence
        sequence = np.sin(freq * t + phase) + noise
        
        # Use first seq_len-1 points as input, last point as target
        X.append(sequence[:-1].reshape(-1, input_size))
        y.append(sequence[-1])
    
    return np.array(X), np.array(y)

# Production deployment example
def main():
    """
    Main function demonstrating production-ready RNN training
    """
    # Set device (GPU if available, else CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # Hyperparameters
    input_size = 1      # Number of input features
    hidden_size = 64    # Size of hidden state
    output_size = 1     # Number of output features
    num_layers = 2      # Number of RNN layers
    seq_len = 49        # Sequence length
    batch_size = 32     # Batch size for training
    learning_rate = 0.001
    epochs = 100
    
    # Create sample data
    print("Generating sample data...")
    X, y = create_sample_data(seq_len + 1, 1000, input_size)
    
    # Convert to PyTorch tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).unsqueeze(1)  # Add dimension for output_size
    
    # Split into train/validation sets
    split_idx = int(0.8 * len(X_tensor))
    X_train, X_val = X_tensor[:split_idx], X_tensor[split_idx:]
    y_train, y_val = y_tensor[:split_idx], y_tensor[split_idx:]
    
    # Create data loaders for batch processing
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model
    print("Initializing model...")
    model = ProductionRNN(input_size, hidden_size, output_size, num_layers).to(device)
    
    # Define loss function and optimizer
    criterion = nn.MSELoss()  # Mean Squared Error for regression
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # Create trainer
    trainer = RNNTrainer(model, criterion, optimizer, device)
    
    # Train the model
    print("Starting training...")
    trainer.train(train_loader, val_loader, epochs, save_path='best_rnn_model.pth')
    
    print("Training completed!")
    
    # Example inference
    model.eval()
    with torch.no_grad():
        # Take first validation sample
        sample_input = X_val[0:1].to(device)  # Shape: (1, seq_len, input_size)
        prediction, _ = model(sample_input)
        actual = y_val[0].item()
        predicted = prediction[0].item()
        
        print(f"\nSample Prediction:")
        print(f"Actual: {actual:.4f}")
        print(f"Predicted: {predicted:.4f}")
        print(f"Error: {abs(actual - predicted):.4f}")

if __name__ == "__main__":
    main()
